## Multi-Agent RAG

### We are going to use wrapper function

In [49]:
!pip install arxiv

#### Fetching data from website using diff langchain tools

<b> 1. Wikipedia tool </b>

In [50]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper   
# top_k_results = fetch only top 1 result, doc_content_chars_max = max no. of char fetched from site
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)
wiki_tool.name

'wikipedia'

In [51]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS   #provided by langchain itself
url = "https://docs.smith.langchain.com/"
loader = WebBaseLoader(url)
web_documents = loader.load()
#chunk_size = atmost size, chunk_overlap = last n words overlappign
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap = 200) 
documents = text_splitter.split_documents(web_documents)
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [52]:
from langchain_community.vectorstores import FAISS   #provided by langchain itself
# Define a minimal embedding-like object using a lambda
class SimpleEmbedding:
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True).tolist()

    def embed_query(self, text):
        return self.model.encode(text, convert_to_numpy=True).tolist()
    
    def __call__(self, text):
        return self.embed_query(text)  # this is what FAISS needs

# Use existing SentenceTransformer model
embedding = SimpleEmbedding(model)

# Use with FAISS
vectordb = FAISS.from_documents(documents=documents[:10], embedding=embedding)
retriever = vectordb.as_retriever()  #a vector store retriever
retriever 

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


VectorStoreRetriever(tags=['FAISS'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000280EF050910>, search_kwargs={})

<b> 2. Retrievel tool </b>


In [53]:
from langchain.tools.retriever import create_retriever_tool
langsmith_tool = create_retriever_tool(
    retriever= retriever, 
    name='langsmith_search', 
    description=  "Search for the information about LangSmith. For any question about langsmith you must use this tool."
)
langsmith_tool.name

'langsmith_search'

<b> 3. Arxiv tool </b>


In [54]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
arxiv_wrapper = ArxivAPIWrapper(top_k_results= 1, doc_content_chars_max=200)
arxiv_tool = ArxivQueryRun(api_wrapper = arxiv_wrapper)
arxiv_tool.name

'arxiv'

In [65]:
#combine all tools
tools = [wiki_tool] 

In [62]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\LANGCHAIN\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200))]

- Now use agent to use all tools in an order 

In [57]:
from dotenv import load_dotenv
load_dotenv()
#Define llama2
from langchain_community.llms import Ollama
llm = Ollama(
    model = "llama2-uncensored",
    temperature = 0.7
)

- Fetchin prompt from langchainhub

In [58]:
from langchain import hub
#get the prompt to use - also we can modify if need.
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [66]:
from langchain.agents import initialize_agent, AgentType
agent = initialize_agent(
    tools= tools,
    llm= llm,
    agent= AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose = True
)

In [68]:
user_prompt = "What is the difference between Machine Learning and Deep Learning?"
response = agent.run(user_prompt)
print(response)



> Entering new AgentExecutor chain...
There are several differences between machine learning (ML) and deep learning (DL), but some of the most notable ones include:
1. Feature representation - In ML, the features used for modeling typically come from hand-crafted feature engineering that is done by humans. However, in DL, the features are automatically extracted from raw data through a process called autoencoders or CNNs. This allows DL models to learn more complex representations of data and make better predictions.
2. Model architecture - ML models typically have simpler architectures with fewer layers than DL models. This is because the complexity of DL models can be overwhelming, and it becomes difficult for humans to understand how these models work in detail. On the other hand, DL models can have hundreds or even thousands of hidden nodes that are connected through nonlinear relationships.
3. Training approach - ML models are typically trained using supervised learning algorith

In [ ]:
!pip install --upgrade langchain langchain-community langchain-core
!pip install --upgrade openapi-python-client


  Using cached ruamel.yaml-0.18.10-py3-none-any.whl.metadata (23 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached typer-0.15.3-py3-none-any.whl.metadata (15 kB)
  Using cached rich-14.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-3.0.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached ruamel.yaml-0.18.10-py3-none-any.whl (117 kB)
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.6 M

In [ ]:
# #Use agent
# toolkit = OpenAPIToolkit(spec=spec, llm=llm)
# agent = create_openapi_agent(llm=llm, toolkit=toolkit, prompt=prompt)
# agent.run("List all available pets in the store")

### Find out github repositery by using github username

In [74]:
!openapi-python-client generate --path github_openapi.json

┌───────────────────── Traceback (most recent call last) ─────────────────────┐
│ d:\LANGCHAIN\venv\Lib\site-packages\openapi_python_client\cli.py:167 in     │
│ generate                                                                    │
│                                                                             │
│   164 │   │   overwrite=overwrite,                                          │
│   165 │   │   output_path=output_path,                                      │
│   166 │   )                                                                 │
│ > 167 │   errors = generate(                                                │
│   168 │   │   custom_template_path=custom_template_path,                    │
│   169 │   │   config=config,                                                │
│   170 │   )                                                                 │
│                                                                             │
│ ┌──────────────────────────────── loca

- First create a json file to fetch repo by username

In [83]:
import requests
url = "https://raw.githubusercontent.com/github/rest-api-description/main/descriptions/api.github.com/api.github.com.json"
response = requests.get(url= url)
json_text = response.content

with open("github_openapi.json", 'wb') as f:
    f.write(json_text)

In [90]:
from langchain_community.utilities.openapi import OpenAPISpec
spec = OpenAPISpec.from_file('github_openapi.json')
spec

Attempting to load an OpenAPI 3.0.3 spec.  This may result in degraded performance. Convert your OpenAPI spec to 3.1.* spec for better support.


AttributeError: 'super' object has no attribute 'parse_obj'

In [89]:
!pip install "pydantic<2.0"

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.2 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 6.0 MB/s eta 0:00:00
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.11.4
    Uninstalling pydantic-2.11.4:
      Successfully uninstalled pydantic-2.11.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires pydantic<3.0.0,>=2.7.4, but you have pydantic 1.10.22 which is incompatible.
langchain-core 0.3.58 requires pydantic<3.0.0,>=2.7.4; python_full_version >= "3.12.4", but you have pydantic 1.10.22 which is incompatible.
langserve 0.3.1 requires pydantic<3.0,>=2.7, but you have pydantic 1.10.22 which is incompatible.
langsmith 0.3.39 requires pydantic<3.0.0,>=2.7.4; python_full_version >= "3.12.4", but you have pydantic 1.10.22 which is incompatible.
openapi-python-client 0.24.3 requires pydantic<3.0.0,>=2.10, but you have pydantic 1.10.22 which is incompatible.
pydantic-settings 2.9.1 requires pydantic>=2.7.0, but you have pydantic 1.10.22 which is incompatible.


In [69]:
from langchain.tools import Tool
def get_repo(username: str) -> str:
    from github_com_client import Client

In [70]:
get_repo('iconic')

ModuleNotFoundError: No module named 'github_com_client'